# DATA TESTING

## Install extra dependencies

In [ ]:
pip install -U pytest
pip install ipywidgets
pip install ipywidgets
pip install fg-data-profiling
pip install great-expectations
#!pip install 'setuptools<81

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from data_profiling import ProfileReport

**NOTE:**

There is currently [an open issue](https://github.com/Data-Centric-AI-Community/fg-data-profiling/issues/1816) where importing ProfileReport from data_profiling results in a `packge not found` error. If the issue persists, it can be mitigated by installing an older version of setuptools. Uncomment the setuptools install line in the extra dependencies block and re-run. 

```text
!pip install 'setuptools<81'
```

See [here](https://github.com/Data-Centric-AI-Community/fg-data-profiling/pull/1819) to track the fix.  

In [ ]:
# Get data from the link. ALternatively, you can use the .csv file from lab-assets from the repo.
csv_url =\
    'http://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'

data = pd.read_csv(csv_url, sep=';')

## Simple inspection

In [ ]:
# with pandas
data.describe()

In [ ]:
# Generate the profile report with Pandas Profiling
profile = ProfileReport(
    data,
    title="Example of summarization of wine data"
)


In [ ]:
profile.to_notebook_iframe()

## Unit tests

### Basic examples - function tests

We will learn how the unit test work on  a simple function. First, we will define a function `square`, which returns the square of a number. Then, we will test it by writing assertions (correct answers) in a test function. 

In [ ]:
import pytest

# install the following to be able to run the tests in notebook
import ipytest
ipytest.autoconfig()

In [ ]:
# A simple function: calculate square of a number
def square(x):
    return x * x

In [ ]:
%%ipytest

# Let's test the function
# Are there any edge cases?
def test_square():
    # FILL IN


Make the test fail to be sure to understand how it works.

### Basic examples - data tests

As we did for the function, we can also write assertions for the data. In the following example we will define a data frame on the fly and thest for the null values in it. 

In [ ]:
%%ipytest

# MAKE THIS FUNCTION FAIL THE TEST

def test_column_is_null():
    df = pd.DataFrame(data = [(1, 0), (2, 1)],
                      columns = ['a', 'b'])
    
    assert np.all(pd.notna(df))

## Test the wine data

Previously, we generated the data frame inside the test function. If we want to run multiple tests on the same df, we would rather pass it to each function as an argument (as usual in programming). To do that in testing, we need to define the data as **fixtures**. They look like ordinary function definitions, preceeded by a decorator `@pytest.fixture`. 

### Raw data tests

In [ ]:
# Define fixtures
@pytest.fixture
def input_schema():
    # Define range and type for each column
    schema = {
    'fixed acidity': {'min': 1.0, 'max': 17.0, 'type': float},
    'volatile acidity': {'min': 0.0, 'max': 2.0, 'type': float},
    'citric acid': {'min': 0.0, 'max': 2.0, 'type': float},
    'residual sugar': {'min': 0.5, 'max': 17.0, 'type': float},
    'chlorides': {'min': 0.0, 'max': 17.0, 'type': float},
    'free sulfur dioxide': {'min': 0.0, 'max': 80.0, 'type': float},
    'total sulfur dioxide': {'min': 0.0, 'max': 300.0, 'type': float},
    'density': {'min': 0.8, 'max': 1.1, 'type': float},
    'pH': {'min': 1.0, 'max': 10.0, 'type': float},
    'sulphates': {'min': 0.0, 'max': 2.0, 'type': float},
    'alcohol': {'min': 7.0, 'max': 17.0, 'type': float},
    'quality': {'min': 1, 'max': 10, 'type': int},
    }
    return schema


# Download the data
@pytest.fixture
def input_data():
    csv_url =\
    'http://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    data = pd.read_csv(csv_url, sep=';')
    return data

Write the following tests:
- is the number of columns in the data frame the same as in schema definition?
- are the values within defined ranges?
- are the types of the columns correct?

In [ ]:
%%ipytest

def test_number_of_columns(input_data, input_schema):
    
    # assert that the column number is the same as the length of the schema


def test_input_data_ranges(input_data, input_schema):
    
    # find min and max value for each column
    # read min and max value for each column from schema

    # for min value of the column: assert that it's always greater or equal than the min from the schema
    # for max value of the column: assert that it's always lesser or equal than the max from the schema
        
        
def test_input_types(input_data, input_schema):
    
    # find the type of each column in the df
    # read the type for each column from schema

    # assert that the type of the column is the same as defined in the schema
    

    

### Feature engineering tests

**NOTE:** Data transformaton should be done only on test dataset. You fit the transformer on the test dataset and then apply it on the train dataset. Since we are only illustrating the functioning of the unit testing, we will do it on the whole dataset.

In [ ]:
from sklearn.preprocessing import StandardScaler
from numpy import mean, std

In [ ]:
# Let's transform a column...

# define standard scaler
scaler = StandardScaler()
# transform data
scaled = scaler.fit_transform(data[['alcohol']])
print(scaled)

In [ ]:
# And check the stats...
print('mean:', mean(scaled))
print('std:', std(scaled))

In [ ]:
@pytest.fixture
def scaled_alcohol():
    csv_url =\
    'http://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    data = pd.read_csv(csv_url, sep=';')
    
    # Define scaler
    scaler = StandardScaler()
    # Transform data
    scaled = scaler.fit_transform(data[['alcohol']])
    return scaled

In [ ]:
%%ipytest
# Test: is mean around zero and std around one?

def test_scaled_mean_zero(scaled_alcohol):
    
    mean_val = mean(scaled_alcohol)
    std_val = std(scaled_alcohol)
    
    # FILL IN: mean arund zero
    # FILL IN: std around one


## Additional exercises:

- implement and Test MinMaxScaler
- test null on 'quality'

## Great Expectations

When you need to test your data automatically and have a lot of additional functionalities from the tool (eg. notifications at the end of the tests), you will use  library, dedicated to data testing. Currently, the most popular one is (Great Expectations)[https://greatexpectations.io/]. Below is a simple demo. 

In [ ]:
# import required libraries
import great_expectations as ge

### Set Up a Data Context

A [Data Context](https://docs.greatexpectations.io/docs/core/set_up_a_gx_environment/create_a_data_context) a defines the storage location for metadata such as information about expectations. It will also contain validation results and associated metadata.  
A `Data Context` can be of two types.

- [File](https://docs.greatexpectations.io/docs/core/set_up_a_gx_environment/create_a_data_context?context_type=file): A persistent Data Context stored in a YAML file.
- [Ephemeral](https://docs.greatexpectations.io/docs/core/set_up_a_gx_environment/create_a_data_context?context_type=ephemeral): A temporary Data Context stored in memory. **Note:** This is the default context if no parameters are passed when calling `get_context()`.


In [ ]:
# Create an ephemeral data context
context = ge.get_context()

# Uncomment the below line to explore the structure of a data context
#print(context)

### Create A Data Source

A [Data Source](https://docs.greatexpectations.io/docs/reference/api/datasource/fluent/datasource_class/) provides a standard API for accessing and interacting with data from a wide variety of source systems such as pandas or spark dataframes, file system data, or SQL data.  

In [ ]:
# name the data source
data_source_name = "pandas_data_source"

# instantiate the data source
data_source_pandas = context.data_sources.add_pandas(name=data_source_name)

### Create A Data Asset

A [Data Asset](https://docs.greatexpectations.io/docs/reference/api/datasource/fluent/dataasset_class/) is a collection of records within a Data Source. It can be used to group validation results. For instance, different Data Assets could be created to group the validation results of differnt parts of a pipeline such as validating training and testing data.

In [ ]:
# name the data asset
data_asset_name = "pandas_data_asset"

# instantiate the data asset
data_asset_pandas = data_source_pandas.add_dataframe_asset(name=data_asset_name)

### Create A Batch Definition

A [Batch Definition](https://docs.greatexpectations.io/docs/core/connect_to_data/dataframes/#create-a-batch-definition) typically defines how the data in a data asset should be retrieved. With dataframes, all of the data in a given dataframe will always be retrieved as a Batch.  

In [ ]:
# Name the batch definition
batch_definition_name = "pandas_batch_definition"

# Instantiate the batch definition
batch_definition_pandas = data_asset_pandas.add_batch_definition_whole_dataframe(
    batch_definition_name
)

### Get the Dataframe Using Batch Parameters

[Dataframes are not stored as part of the as part of the Batch Definition or Data Asset](https://docs.greatexpectations.io/docs/core/connect_to_data/dataframes/#provide-a-dataframe-through-batch-parameters). Instead, they are passed in at runtime as a Batch Parameter  dictionary.



In [ ]:
# Set the batch paramters as the wine df used earlier in the lab
batch_parameters_wine_data = {'dataframe': data }

# Use the batch definition to get the data
batch = batch_definition_pandas.get_batch(batch_parameters_wine_data)

### Create the Expectations 

[Expectations](https://docs.greatexpectations.io/docs/core/define_expectations/create_an_expectation) are the actual tests or verifiable assertions for the data. Three examples of passing expectations are defined below. Additionally, an expectation that is expected to fail is defined to demonstrate a failing test.

In [ ]:
# Define the expectations
expectation1 = ge.expectations.ExpectColumnToExist(column="pH")
expectation2 = ge.expectations.ExpectColumnValuesToNotBeNull(column='quality')
expectation3 = ge.expectations.ExpectColumnValuesToBeOfType(column='chlorides', type_='float')

In [ ]:
# Set the last expectation to something that should fail and note the difference in output
expectation4 = ge.expectations.ExpectColumnValuesToBeOfType(column='chlorides', type_='str')

### Validate the Data

[Data is tested](https://docs.greatexpectations.io/docs/core/define_expectations/test_an_expectation) by using the `batch.validate` method. See [batch documentation](https://docs.greatexpectations.io/docs/reference/api/datasource/fluent/interfaces/batch_class/#methods)

In [ ]:
# Validate the passing expectations and view the results
validate_res1 = batch.validate(expectation1)
validate_res2 = batch.validate(expectation2)
validate_res3 = batch.validate(expectation3)

print(validate_res1)
print(validate_res2)
print(validate_res3)

In [ ]:
# Run the failing validation and view the reuslts to note the difference
validate_res4 = batch.validate(expectation4)
print(validate_res4)

### BONUS

- create and vailidate a few extra Expectations for the wine data using the above examples
- add data testing to a personal/school project